In [0]:
import requests
import urllib3
import pandas as pd
import logging
import os
import time
import json
from datetime import datetime
from pathlib import Path
import configparser


def ensure_file_exists(path: str | Path) -> Path:
    """Ensure the config file exists and return a Path object."""
    p = Path(path).expanduser().resolve()
    if not p.exists():
        raise FileNotFoundError(f"Configuration file not found: {p}")
    return p


def load_json_config(path: str | Path) -> dict:
    p = ensure_file_exists(path)
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)


def main():
    print("Hello World!")
    Current_path=os.getcwd()
    config_file =Current_path+ "/config.json"
    print(config_file)
    ensure_file_exists(config_file)
    config = load_json_config(config_file)
    print(config.get('SAP_BO_Credentials',{}))
    fileName=config.get('Static_Dict',{}).get('Report_ID_List')
    Report_fileName=Current_path+"/"+fileName
    print(Report_fileName)
    # Read the tab-delimited file
    ensure_file_exists(Report_fileName)
    df = pd.read_csv(Report_fileName, delimiter='\t')  # Use '\t' for tab-delimited files
    print(df.head())


if __name__ == "__main__":
    main()

In [0]:
%sh

ls -l /Workspace/Users/baodi.wilkinson.external@atradius.com/dataplatform-bo-migration/non_active-schedules | wc -l
# grep -v '\.json$'
#  wc -l
# grep "(" 
# 

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_doc_global_similarity where src_document_id in ('10123570','10121894');
SELECT canonical_id, COUNT(distinct Document_Id) AS dup_count
FROM dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_similarity
-- WHERE similarity_flag = 'LIKELY_DUPLICATE'
GROUP BY canonical_id
ORDER BY dup_count DESC
LIMIT 50;

%sql
-- dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_silver
select count(distinct Document_Id) from dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_silver;
SELECT * FROM dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_similarity WHERE canonical_id in ('awtg1dl.erdaozty8402yfm','axndeeg.fczknaly6qmhlyy');

In [0]:
%sql
-- select cluster_id, count(distinct Document_Id) as cluster_size 
-- from dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_doc_clusters_v25 
-- -- where cluster_size > 5 
-- group by cluster_id order by cluster_size desc;

-- select Document_Id, count(distinct cluster_id) as cluster_count from dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_doc_clusters_v25 group by Document_Id order by cluster_count desc limit 100;

-- dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_doc_clusters
--  group by  cluster_id order by count( Document_Id) desc

select cluster_ed.cluster_id, cms.Document_Id, cms.Full_path, cms.DataSource_Name
from dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_doc_clusters_v25 as cluster_ed
left join
dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_silver as cms
on cluster_ed.Document_Id = cms.Document_Id
where cluster_ed.cluster_id=365347

In [0]:
%sql
select  count(distinct Document_Id, Data_Provider_ID) as doc_count
-- from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.webi_metadata_cms;
-- distinct Document_Id, Document_name, Full_path,  Data_Profider_Refresh_Time, updated  
from dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_metadata_cms_silver
-- from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.webi_metadata_cms
where 
Data_Profider_Refresh_Time > '2022-01-01'
and 
updated > '2022-01-01' 
limit 100


In [0]:
%sql
select DataSource_Name, count(distinct Document_Id) as doc_count
from _sqldf
group by DataSource_Name
order by doc_count desc